In [ ]:
#All the imports
import h5py
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d
from scipy.stats import norm
from scipy.stats import kurtosis
from scipy.ndimage import maximum_filter, label
import multiprocessing as mp
import os
import time

# shared code and paths (see README.md and shared/)
import sys
from pathlib import Path
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "shared" / "paths.py").exists())
sys.path.insert(0, str(REPO))
from shared.fields import partial_x_fft, partial_y_fft
from shared.filtering import gaussian_fft_filter_2d
from shared.paths import DATA_ROOT, NAS_ROOT, EXTERNAL_DATA_ROOT, results_dir, cache_dir as shared_cache_dir
RESULTS = results_dir("quality_checks", "Jrms")

In [ ]:
# List of simulation directories
sim_dirs = [
    f'{DATA_ROOT}/beta_p_1/beta_e_1/'
]

dx = 0.0625
dy = 0.0625

In [ ]:
def process_timestep(args):
    sim_dir, i = args
    
    # print(f"{sim_dir}: i = {i}", flush=True)

    File_Bx = f'{sim_dir}/Bx_ApJ_t{i}.h5'
    File_By = f'{sim_dir}/By_ApJ_t{i}.h5'

    with h5py.File(File_Bx, 'r') as fBx, h5py.File(File_By, 'r') as fBy:
        data_Bx = fBx['DS1'][:].T
        data_By = fBy['DS1'][:].T

    # Filtering 
    
    data_Bx_filtered = gaussian_fft_filter_2d(data_Bx, k0=0.2 * len(data_Bx))
    data_By_filtered = gaussian_fft_filter_2d(data_By, k0=0.2 * len(data_Bx))

    data_Bx = data_Bx_filtered
    data_By = data_By_filtered
    
    J_par_FT = partial_x_fft(data_By, dx) - partial_y_fft(data_Bx, dy)
    rms = np.sqrt(np.mean(J_par_FT**2))

    return i, rms


def process_sim_dir(sim_dir):
    timesteps = [(sim_dir, i) for i in range(300)]

    start = time.perf_counter()
    with mp.Pool(processes=128) as pool:
        results = pool.map(process_timestep, timesteps)
    end = time.perf_counter()

    print(f"Total runtime: {end - start:.2f} seconds")
    # sort by timestep
    results.sort()
    J_rms = [rms for _, rms in results]

    return sim_dir, J_rms


if __name__ == "__main__":
    all_J_rms_filter = {}

    for sim_dir in sim_dirs:
        sim_dir, J_rms = process_sim_dir(sim_dir)
        all_J_rms_filter[sim_dir] = J_rms

In [ ]:
timesteps = np.arange(0, 300)

plt.figure(figsize=(10, 6))

# No filter
# for sim_dir, J_rms in all_J_rms_nofilter.items():
#     label = sim_dir.split('/')[-1] + " (no filter)"
#     plt.plot(timesteps, J_rms, label=label)

# Filtered
for sim_dir, J_rms in all_J_rms_filter.items():
    label = sim_dir.split('/')[-1] + " (filtered)"
    plt.plot(timesteps, J_rms, '--', label=label)  # dashed for filtered

plt.xlabel('Timestep')
plt.ylabel(r'$J_{rms}$')
plt.title('J_rms vs Time for Different Simulations')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(RESULTS / "Jrms.png", dpi=300)
plt.show()

In [ ]:
np.max(J_rms)

In [ ]:
import numpy as np

J_rms = np.array(J_rms)
timesteps = np.array(timesteps)

J_max = np.max(J_rms)
threshold = 0.95 * J_max

indices = np.where(J_rms >= threshold)[0]
times_above = timesteps[indices]
values_above = J_rms[indices]

In [ ]:
print(indices)